# Notebook G — RecBole sequential baselines under LOO

Runs the sequential baselines (SASRec, BERT4Rec, CORE, NARM, Caser, SRGNN, STAMP) under the **same LOO split** as CaST-POI, via `run_recbole.py` (RecBole benchmark mode on the exported `data_loo`; each model uses its own published RecBole default lr). `run_recbole.py` reorders ranks into CaST-POI's test order by `check_ins_id`, so every baseline is directly comparable to CaST-POI on the identical LOO test set. **All baselines are run by us under one pipeline -- none are cited numbers.**

Verified locally: `build_samples` on `data_loo` yields the LOO test set (1,044 / 2,280 / 3,956, one per user), aligned to CaST-POI. Run the CaST-POI notebooks first (or this rebuilds `data_loo` identically -- it is deterministic). ~63 runs; RecBole seq models are fast on small POI data.

In [ ]:
# # === Setup: Drive, workdir, data ===
# import os, sys, json, glob, subprocess, itertools
# try:
#     from google.colab import drive; drive.mount('/content/drive'); ON_COLAB = True
# except Exception:
#     ON_COLAB = False

# CACHE = '/content/drive/MyDrive/castpoi'          # <- your Drive folder (has data_official/)
# WORK  = '/content/work' if ON_COLAB else os.getcwd()
# os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
# if WORK not in sys.path: sys.path.insert(0, WORK)

# RUNS = f'{CACHE}/runs' if ON_COLAB else 'runs'    # results persist to Drive
# DATA = 'data_official'
# if ON_COLAB:
#     assert os.path.isdir(f'{CACHE}/data_official'), \
#         "no data_official on Drive at %s/data_official" % CACHE
#     os.makedirs(DATA, exist_ok=True)
#     subprocess.run(f'cp -rn {CACHE}/data_official/* {DATA}/', shell=True)
# os.makedirs('castpoi', exist_ok=True)
# print('ON_COLAB', ON_COLAB, '| data_official present:', os.path.isdir(DATA),
#       '| datasets:', sorted(os.listdir(DATA)) if os.path.isdir(DATA) else None)


In [ ]:
# === Setup: Drive, workdir, data ===
import os, sys, json, glob, shutil, subprocess, itertools

CACHE = '/content/drive/MyDrive/castpoi'          # <- your Drive folder (has data_official/)

# 1) 是否在 Colab 由 import 决定。能 import 却挂载失败 = 授权没走完,必须抛;
#    原来的 try/except 会静默降级成 ON_COLAB=False,于是 RUNS 变成裸的 'runs'
#    (临时盘)、data_official 根本不拷,一路跑到 leaderboard 才以全 nan 暴露。
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')                # 不要包 try/except
    assert os.path.isdir(CACHE), f'Drive 已挂载但找不到 {CACHE}'
ON_COLAB = IN_COLAB

WORK = '/content/work' if ON_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
if WORK not in sys.path: sys.path.insert(0, WORK)

RUNS = f'{CACHE}/runs' if ON_COLAB else 'runs'   # results persist to Drive
DATA = 'data_official'

# 2) 先删后拷,不用 cp -rn。-n 会跳过已存在的文件,上次拷到一半留下的
#    截断 csv 就永远换不掉(tky 那次 UTCTimeOffset 解析失败即此)。
if ON_COLAB:
    src = f'{CACHE}/data_official'
    assert os.path.isdir(src), f'no data_official on Drive at {src}'
    shutil.rmtree(DATA, ignore_errors=True)
    assert not os.path.exists(DATA), f'{DATA} 删不掉,先手动清理'
    subprocess.run(['cp', '-r', src, DATA], check=True)   # check=True:拷贝失败立即抛

# 3) 逐文件核字节数,挡住"返回码 0 但内容短了"的静默截断
EXPECT = {
    'nyc/train_sample.csv':               9261516,
    'nyc/validate_sample_with_traj.csv':  1062922,
    'nyc/test_sample_with_traj.csv':      1022866,
    'tky/train_sample.csv':              34916166,
    'tky/validate_sample_with_traj.csv':  4151575,
    'tky/test_sample_with_traj.csv':      4102859,
    'ca/train_sample.csv':               26601143,
    'ca/validate_sample_with_traj.csv':   2559600,
    'ca/test_sample_with_traj.csv':       1788518,
}
missing = [k for k in EXPECT if not os.path.isfile(f'{DATA}/{k}')]
assert not missing, f'staging 缺文件: {missing}'
bad = {k: (os.path.getsize(f'{DATA}/{k}'), v) for k, v in EXPECT.items()
       if os.path.getsize(f'{DATA}/{k}') != v}
assert not bad, f'staging 副本损坏 {{文件: (实际, 期望)}}: {bad}'

# 4) 清掉上一轮可能残留的半成品 data_loo,避免下游静默用到旧 split
shutil.rmtree('data_loo', ignore_errors=True)

os.makedirs('castpoi', exist_ok=True)
print(f'ON_COLAB {ON_COLAB} | WORK {WORK}')
print(f'RUNS     {RUNS}')      # <- 必须是 Drive 路径;裸的 'runs' 说明挂载没成
print(f'data_official OK: {sorted(os.listdir(DATA))} — 9 个文件字节数全部匹配')

Mounted at /content/drive
ON_COLAB True | WORK /content/work
RUNS     /content/drive/MyDrive/castpoi/runs
data_official OK: ['ca', 'nyc', 'tky'] — 9 个文件字节数全部匹配


In [ ]:
# === Install RecBole (same version as the chronological campaign) ===
!pip -q install recbole==1.2.1 --no-deps 2>&1 | tail -1
# Colab pre-installs scikit-learn/tensorboard/scipy/pandas, but pin them explicitly so a
# drifted environment does not fail at `import recbole` (verified needed 2026-07-24).
!pip -q install colorlog==4.7.2 colorama thop tabulate texttable kmeans_pytorch scikit-learn tensorboard 2>&1 | tail -1
import recbole; print('recbole', recbole.__version__, 'ready')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.3 MB/s eta 0:00:00
recbole 1.2.1 requires colorama==0.4.4, but you have colorama 0.4.6 which is incompatible.
recbole 1.2.1 ready


### Write the package + `run_experiment.py` + `run_recbole.py` + `build_loo_split.py`

In [ ]:
%%writefile castpoi/__init__.py
"""CaST-POI."""


Writing castpoi/__init__.py


In [ ]:
%%writefile castpoi/config.py
"""Model and training configuration.

One BASE_CONFIG is used for every dataset; there are no per-dataset defaults.
Preprocessing and splitting are not configurable here. They come from the
official files loaded by official.py and from build_loo_split.py; see DATA.md.
"""
import copy
from typing import Any, Dict

BASE_CONFIG: Dict[str, Any] = {
    # Model
    "poi_embed_dim": 128,
    "slot_embed_dim": 16,
    "spatial_dim": 32,
    "dist_embed_dim": 16,
    "num_dist_buckets": 8,
    # Revisit gate: MLP width, and the window the visit counts are taken over.
    # The sequence encoder still sees only max_history_len; counting is cheap,
    # so this window can be longer.
    "repeat_gate_hidden": 32,
    "repeat_history_len": 512,

    # Training objective. "sampled" scores the positive against `num_negatives`
    # sampled negatives; "full" scores it against the entire POI vocabulary.
    # Evaluation is always full-vocabulary, and the RecBole baselines train with
    # full-vocabulary cross-entropy, so run.py sets "full". The vocabulary is
    # small enough (4,980 / 7,832 / 9,689) for a dense softmax.
    "train_objective": "sampled",   # "sampled" | "full"
    "num_negatives": 499,           # ignored when train_objective == "full"
    "batch_size": 512,
    "eval_batch_size": 1024,
    "num_epochs": 50,
    "learning_rate": 2e-3,
    "weight_decay": 1e-4,
    "dropout": 0.1,
    "gradient_clip": 5.0,
    "early_stopping_patience": 10,
    "warmup_epochs": 3,
    "label_smoothing": 0.02,
    "explore_weight": 1.5,
    "bpr_weight": 0.5,
    "bpr_margin": 1.0,

    # Model input truncation, not a data filter.
    "max_history_len": 50,
    # Selects the validation metric. run.py widens this to [1, 5, 10, 20] so the
    # saved metrics cover every K the tables use.
    "eval_ks": [5, 10],
}

DATASETS = ("nyc", "tky", "ca")


def resolve_config(dataset: str, overrides: Dict[str, Any] = None) -> Dict[str, Any]:
    """Effective config for `dataset`. Identical for every dataset by default."""
    ds = dataset.lower()
    if ds not in DATASETS:
        raise ValueError(f"unknown dataset {dataset!r}; choose from {list(DATASETS)}")
    cfg = copy.deepcopy(BASE_CONFIG)
    if overrides:
        cfg.update({k: v for k, v in overrides.items() if v is not None})
    cfg["dataset"] = ds

    for gone in ("split_protocol", "test_size", "min_poi_checkins", "min_user_checkins"):
        if gone in cfg:
            raise ValueError(
                f"{gone!r} is not a configuration option: preprocessing comes from "
                f"the official STHGCN/LLM4POI output, not from this package. "
                f"See DATA.md.")
    return cfg


Writing castpoi/config.py


In [ ]:
%%writefile castpoi/timeparse.py
"""Time handling for the LLM4POI preprocessed files.

`UTCTimeOffsetEpoch` is unusable: it is `UTCTimeOffset` passed through a naive
`datetime.timestamp()` on a UTC+10/+11 machine, so the same offset appears for
New York, Tokyo and California alike, and hour-of-day read from it places Tokyo's
quietest hour at 17:00.

`UTCTimeOffset` is used instead. Its meaning differs per dataset:
  nyc, tky : local wall clock, offset applied (Foursquare).
  ca       : raw UTC, offset not applied (Gowalla).

`assert_human_rhythm` checks the result: under the correct reading each city
shows a 3-5am trough and a daytime peak.

Two quantities come out of this module and are not interchangeable:
  ts_utc       absolute instant, for elapsed time between check-ins.
  hour / dow   local calendar position, for the periodic features.
"""
from typing import Dict, Tuple

import numpy as np
import pandas as pd

# tz            IANA zone of the city.
# column_means  what the `UTCTimeOffset` column actually holds.
TIME_SPEC: Dict[str, Dict[str, str]] = {
    "nyc": {"tz": "America/New_York", "column_means": "local"},
    "tky": {"tz": "Asia/Tokyo", "column_means": "local"},
    "ca": {"tz": "America/Los_Angeles", "column_means": "utc"},
}


class TimeParseError(RuntimeError):
    pass


def parse_times(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Add ts_utc (int seconds), local_hour (float 0-24), local_dow (0=Mon).

    `UTCTimeOffsetEpoch` is ignored entirely. It is never read by this package.
    """
    ds = dataset.lower()
    if ds not in TIME_SPEC:
        raise TimeParseError(f"no time spec for dataset {ds!r}")
    spec = TIME_SPEC[ds]

    if "UTCTimeOffset" not in df.columns:
        raise TimeParseError(
            f"{ds}: column 'UTCTimeOffset' is missing. The corrupt "
            f"'UTCTimeOffsetEpoch' column is not an acceptable substitute.")

    naive = pd.to_datetime(df["UTCTimeOffset"])
    if naive.isna().any():
        raise TimeParseError(f"{ds}: {naive.isna().sum()} unparseable timestamps")

    if spec["column_means"] == "local":
        # Wall clock in the city. Localize to recover the instant. DST fall-back
        # hours are ambiguous; resolve to standard time and shift nonexistent
        # spring-forward times rather than dropping check-ins.
        aware_local = naive.dt.tz_localize(spec["tz"], ambiguous=False, nonexistent="shift_forward")
        utc = aware_local.dt.tz_convert("UTC")
        local = aware_local
    else:
        utc = naive.dt.tz_localize("UTC")
        local = utc.dt.tz_convert(spec["tz"])

    out = df.copy()
    out["ts_utc"] = (utc.astype("int64") // 10 ** 9).astype("int64")
    out["local_hour"] = (local.dt.hour + local.dt.minute / 60.0 + local.dt.second / 3600.0).astype("float32")
    out["local_dow"] = local.dt.dayofweek.astype("int8")  # 0=Monday
    return out


def rhythm_stats(local_hour: np.ndarray) -> Dict[str, float]:
    """Shape of the daily check-in rhythm."""
    h = np.asarray(local_hour)
    dist = np.bincount(h.astype(int).clip(0, 23), minlength=24) / max(len(h), 1)
    night = float(dist[3:6].sum())
    day = float(dist[11:22].sum())
    return {
        "night_3_6_frac": night,
        "day_11_22_frac": day,
        "day_night_ratio": day / max(night, 1e-9),
        "argmin_hour": int(dist.argmin()),
        "argmax_hour": int(dist.argmax()),
        "hist": dist.tolist(),
    }


def assert_human_rhythm(local_hour: np.ndarray, dataset: str, min_ratio: float = 3.0) -> Dict[str, float]:
    """Raise if the parsed local time is not a plausible human rhythm.

    A guard, not a formality. Every wrong reading of these columns that we found
    puts the daily minimum somewhere between 11:00 and 17:00. Real check-ins
    trough between 03:00 and 06:00.
    """
    st = rhythm_stats(local_hour)
    bad_min = not (1 <= st["argmin_hour"] <= 7)
    bad_ratio = st["day_night_ratio"] < min_ratio
    if bad_min or bad_ratio:
        raise TimeParseError(
            f"{dataset}: parsed local time does not look like human behaviour "
            f"(quietest hour = {st['argmin_hour']:02d}:00, busiest = {st['argmax_hour']:02d}:00, "
            f"day/night ratio = {st['day_night_ratio']:.1f}). Expected the trough between "
            f"03:00 and 06:00. The time column is being read wrong; check TIME_SPEC[{dataset!r}].")
    return st


Writing castpoi/timeparse.py


In [ ]:
%%writefile castpoi/utils.py
"""Seeding, environment capture, logging, and artifact IO.

Every run records enough environment metadata to prove where and when it ran.
"""
import json
import logging
import os
import platform
import random
import socket
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def pick_device(requested: str = "auto") -> torch.device:
    """auto -> cuda if present, else cpu.

    MPS is deliberately NOT chosen automatically. It is available on Apple
    silicon and this model trains to a NaN loss on it while CPU and CUDA train
    normally; the cause has not been isolated. Since a NaN loss used to surface
    as a perfect 100% score, an automatic device that quietly breaks training is
    the last thing this package should do. Pass --device mps explicitly if you
    want to investigate it.
    """
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def _git_commit() -> Optional[str]:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=Path(__file__).resolve().parent,
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip() or None
    except Exception:
        return None


def env_info() -> Dict[str, Any]:
    """Provenance block embedded in every result file."""
    info = {
        "hostname": socket.gethostname(),
        "platform": platform.platform(),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "numpy": np.__version__,
        "cuda_available": torch.cuda.is_available(),
        "git_commit": _git_commit(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "argv": sys.argv,
        "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_count"] = torch.cuda.device_count()
    return info


def setup_logger(log_path: Path, name: str = "castpoi") -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")

    fh = logging.FileHandler(log_path, mode="a")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger


class _Encoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, Path):
            return str(o)
        return super().default(o)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, cls=_Encoder)
    tmp.replace(path)  # atomic: a killed job never leaves a half-written result


def read_json(path: Path) -> Any:
    with open(path) as f:
        return json.load(f)


def append_jsonl(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(obj, cls=_Encoder) + "\n")


def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


Writing castpoi/utils.py


In [ ]:
%%writefile castpoi/metrics.py
"""Full-vocabulary ranking metrics.

Metrics are returned per sample, so the reported mean and any paired test are
computed from the same array.
"""
from typing import Dict, List, Sequence

import numpy as np

# K values written to metrics.json, independent of `eval_ks`, which only selects
# the validation metric.
REPORT_KS = (1, 5, 10, 20)
import torch


def per_sample_ranks(scores: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """1-indexed rank of the target under full-vocabulary scoring.

    Ties use the mid-rank convention

        rank = 1 + #{strictly higher} + #{tied} / 2

    rather than 1 + #{strictly higher}. The optimistic rule is only equivalent
    when scores are dense: a count-based scorer assigns exactly 0 to every
    unvisited POI, so an unvisited target ties with thousands of items and would
    be credited with a near-top rank. Dense neural scorers are unaffected, so the
    optimistic rule would favour whichever method produces more ties.

    Non-finite scores raise instead of ranking, because every comparison with NaN
    is False and a diverged model would otherwise be scored as rank 1 everywhere,
    i.e. a perfect result.
    """
    if not torch.isfinite(scores).all():
        n_bad = int((~torch.isfinite(scores)).sum())
        raise ValueError(
            f"{n_bad} of {scores.numel()} scores are NaN or Inf. Ranking them would "
            f"report rank 1 for every sample (every comparison with NaN is False), "
            f"i.e. a perfect 100% score from a broken model. Check for a diverged "
            f"loss or an unsupported device dtype.")
    tgt = scores.gather(1, targets.unsqueeze(1))
    greater = (scores > tgt).sum(1)
    tied = (scores == tgt).sum(1) - 1          # exclude the target itself
    return greater + 1 + tied.float() / 2.0


def metrics_from_ranks(ranks: np.ndarray, ks: Sequence[int] = (5, 10, 20)) -> Dict[str, np.ndarray]:
    """Per-sample metric vectors. Mean of each vector is the reported number."""
    out: Dict[str, np.ndarray] = {}
    for k in ks:
        out[f"HR@{k}"] = (ranks <= k).astype(np.float64)
        out[f"NDCG@{k}"] = np.where(ranks <= k, 1.0 / np.log2(ranks + 1.0), 0.0)
    out["MRR"] = 1.0 / ranks
    return out


def summarize(per_sample: Dict[str, np.ndarray]) -> Dict[str, float]:
    return {k: float(v.mean()) for k, v in per_sample.items()}


def format_metrics(m: Dict[str, float], pct: bool = True) -> str:
    order = sorted(m.keys(), key=lambda s: (s.split("@")[0], int(s.split("@")[1]) if "@" in s else 0))
    return " | ".join(f"{k}: {m[k] * 100:.2f}" if pct else f"{k}: {m[k]:.4f}" for k in order)




Writing castpoi/metrics.py


In [ ]:
%%writefile castpoi/official.py
"""Load the official STHGCN/LLM4POI splits.

`data_official/<ds>/` holds the output of LLM4POI's own preprocessing, run
unmodified; see DATA.md. The regenerated `train_sample.csv` is byte-identical to
the published `w11wo/LLM4POI` files for all three datasets, apart from 424
mojibake rows in that upload.

The properties below come from the official pipeline, not from this module:
min_poi_freq / min_user_freq of 9 / 9 with `count > freq` semantics (CA is
filtered twice; NYC arrives pre-split from GETNext), a global chronological
80/10/10 split over all check-ins, 24-hour session gaps with singleton sessions
dropped, and removal of val/test rows whose user or POI is absent from train.

Two things are done here because the official files leave them:

1. `UTCTimeOffsetEpoch` is corrupt in every published file (see timeparse.py), so
   `UTCTimeOffset` is parsed instead.
2. POI id 0 is a real POI in their label encoding, while the model reserves 0 for
   padding, so ids are shifted where needed.
"""
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from .timeparse import TIME_SPEC, assert_human_rhythm, parse_times

DEFAULT_OFFICIAL = Path(__file__).resolve().parent.parent / "data_official"

FILES = {
    "train": "train_sample.csv",
    "val": "validate_sample_with_traj.csv",
    "test": "test_sample_with_traj.csv",
}

SOURCE = {
    "nyc": {"note": "Foursquare New York City", "platform": "Foursquare",
            "provenance": "GETNext pre-split NYC_{train,val,test}.csv"},
    "tky": {"note": "Foursquare Tokyo", "platform": "Foursquare",
            "provenance": "STHGCN filter(9,9) + global chronological 80/10/10"},
    "ca": {"note": "Gowalla California", "platform": "Gowalla",
           "provenance": "STHGCN filter(9,9) applied TWICE + global chronological 80/10/10"},
}

# The official pipeline label-encodes ids under two conventions: NYC yields
# ids 0..N-1 with padding N, while TKY and CA yield ids 1..N with padding 0. So
# PoiId==0 is a real POI in NYC and the padding bucket in TKY/CA. Treating them
# alike would count the padding bucket as an entity and, given that 0 is reserved
# for sequence padding here, would leave a dead embedding row in TKY/CA. The
# convention is declared explicitly rather than inferred from min(), so a split
# that happens not to contain id 0 cannot be misread.
ID_CONVENTION = {"nyc": "zero_indexed", "tky": "one_indexed", "ca": "one_indexed"}


class OfficialDataMissing(RuntimeError):
    pass


def _read(ds: str, root: Path) -> Dict[str, pd.DataFrame]:
    d = root / ds
    out = {}
    for split, fn in FILES.items():
        p = d / fn
        if not p.exists():
            raise OfficialDataMissing(
                f"{p} not found. Regenerate with LLM4POI's own pipeline; see "
                f"DATA.md. This code will not "
                f"substitute its own preprocessing.")
        out[split] = pd.read_csv(p, low_memory=False)
    return out


def poi_shift(ds: str) -> int:
    """Offset mapping the official PoiId onto our vocabulary, where 0 = padding.

    zero_indexed (nyc): official ids 0..N-1 -> 1..N, shift +1.
    one_indexed  (tky, ca): official ids are already 1..N with 0 as their own
                            padding bucket -> no shift.
    """
    return 1 if ID_CONVENTION[ds] == "zero_indexed" else 0


def real_id_range(ds: str, train: pd.DataFrame, col: str) -> Tuple[int, int]:
    """(n_distinct_real, our_max_index) for a label-encoded column."""
    if ID_CONVENTION[ds] == "zero_indexed":
        n = int(train[col].max()) + 1        # 0..max are all real; padding is max+1
    else:
        n = int(train[col].max())            # 1..max are real; 0 is padding
    return n, n


def _trajectories(df: pd.DataFrame, ds: str) -> Dict[int, List[Dict]]:
    df = parse_times(df, ds)
    # mergesort = stable, so exact (UserId, ts_utc) ties keep their file order
    # instead of pandas' quicksort tie-break. TKY ships 467 exact duplicate
    # (UserId, ts_utc, PoiId) rows and CA 35; with an unstable sort, a duplicate
    # of the target could land in the *history* of its own eval sample. Only 36
    # samples out of 60k were affected, so this changes no conclusion, but it is
    # the difference between "causal" and "causal except when pandas feels like it".
    df = df.sort_values(["UserId", "ts_utc"], kind="mergesort")
    has_cat = "PoiCategoryName" in df.columns
    # check_ins_id rides along unused by the model. It is the only stable join key
    # back to a foreign implementation's test set: the official pipeline assigns it
    # (preprocess_main.py:45) as a rank over UTCTimeOffset, a wall-clock string, so
    # it survives the timezone corruption that makes UTCTimeOffsetEpoch unusable
    # across machines. Without it, comparing against a re-run of STHGCN means
    # replaying this function's grouping from outside and hoping it stays in sync.
    cols = ["UserId", "PoiId", "ts_utc", "local_hour", "local_dow",
            "Latitude", "Longitude", "pseudo_session_trajectory_id", "check_ins_id"]
    if has_cat:
        cols.append("PoiCategoryName")

    shift = poi_shift(ds)
    trajs: Dict[int, List[Dict]] = {}
    for row in df[cols].itertuples(index=False, name=None):
        trajs.setdefault(int(row[0]), []).append({
            "poi_idx": int(row[1]) + shift,      # 0 reserved for padding
            "ts_utc": float(row[2]),
            "hour": float(row[3]),
            "dow": int(row[4]),
            "latitude": float(row[5]),
            "longitude": float(row[6]),
            "traj_id": int(row[7]),
            "check_ins_id": int(row[8]),
            "category": row[9] if has_cat else "Unknown",
        })
    return trajs


def fingerprint(splits: Dict[str, Dict], num_pois: int) -> str:
    h = hashlib.sha256()
    h.update(f"OFFICIAL_V1|{num_pois}".encode())
    for name in ("train", "val", "test"):
        h.update(f"|{name}|".encode())
        for uid in sorted(splits[name]):
            h.update(f"{uid}:".encode())
            h.update(np.array([c["poi_idx"] for c in splits[name][uid]], dtype=np.int64).tobytes())
            h.update(np.array([c["ts_utc"] for c in splits[name][uid]], dtype=np.int64).tobytes())
    return h.hexdigest()[:16]


def load_official(ds: str, root: Path = None) -> Dict:
    ds = ds.lower()
    root = Path(root or DEFAULT_OFFICIAL)
    info = SOURCE[ds]
    print(f"\n{'=' * 68}\n[official] {ds.upper()} ({info['note']}, {info['platform']})\n"
          f"[official] provenance: {info['provenance']}\n{'=' * 68}")

    raw = _read(ds, root)
    for k, df in raw.items():
        print(f"[official] {k:5s}: {len(df):>7,} check-ins  "
              f"({(root / ds / FILES[k]).stat().st_size / 1e6:.1f} MB)")

    # Vocabulary comes from TRAIN, exactly as their id_encode does. val/test have
    # already had unseen users and POIs removed upstream.
    shift = poi_shift(ds)
    n_train_pois, max_idx = real_id_range(ds, raw["train"], "PoiId")
    n_users, _ = real_id_range(ds, raw["train"], "UserId")
    num_pois = max_idx + 1                           # index 0 is our padding

    for k in ("val", "test"):
        lo, hi = raw[k]["PoiId"].min() + shift, raw[k]["PoiId"].max() + shift
        assert 1 <= lo and hi <= max_idx, (
            f"{ds}/{k} PoiId maps outside 1..{max_idx} (got {lo}..{hi}); "
            f"unseen POIs should have been removed upstream")

    splits = {k: _trajectories(df, ds) for k, df in raw.items()}
    used = {c["poi_idx"] for s in splits.values() for t in s.values() for c in t}
    assert 0 not in used, f"{ds}: padding index 0 leaked into the data"
    dead = set(range(1, num_pois)) - used
    if dead:
        print(f"[official] note: {len(dead)} vocabulary slots never appear in any split")

    all_df = parse_times(pd.concat(raw.values()), ds)
    st = assert_human_rhythm(all_df["local_hour"].values, ds)

    # POI metadata is aggregated over the train split only, so no statistic is
    # computed over test. POIs absent from train keep coordinate (0,0) and
    # category Unknown; their embeddings never receive a gradient.
    train_df = all_df[all_df["_split"] == "train"] if "_split" in all_df.columns else \
        parse_times(raw["train"], ds)
    poi_locations = np.zeros((num_pois, 2), dtype=np.float64)
    poi_categories: Dict[int, str] = {}
    agg = {"Latitude": "mean", "Longitude": "mean"}
    if "PoiCategoryName" in train_df.columns:
        agg["PoiCategoryName"] = lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
    for pid, row in train_df.groupby("PoiId").agg(agg).iterrows():
        i = int(pid) + shift
        if not 1 <= i < num_pois:
            continue                                 # the official padding bucket
        poi_locations[i] = [row["Latitude"], row["Longitude"]]
        poi_categories[i] = row.get("PoiCategoryName", "Unknown")

    counts = np.zeros(num_pois, dtype=np.float64)
    for traj in splits["train"].values():
        for c in traj:
            counts[c["poi_idx"]] += 1
    counts[0] = 0.0
    popularity = counts / counts.sum()

    n_check = sum(len(df) for df in raw.values())
    n_traj = int(pd.concat(raw.values())["pseudo_session_trajectory_id"].nunique())
    stats = {
        "dataset": ds.upper(), "platform": info["platform"], "note": info["note"],
        "provenance": info["provenance"], "id_convention": ID_CONVENTION[ds],
        "users": n_users, "pois": n_train_pois, "checkins": n_check,
        "trajectories": n_traj,
        "sparsity_pct": 100 * (1 - n_check / (n_users * n_train_pois)),
        "avg_traj_len": n_check / n_traj,
        "train_checkins": len(raw["train"]), "val_checkins": len(raw["val"]),
        "test_checkins": len(raw["test"]),
        "train_users": len(splits["train"]), "test_users": len(splits["test"]),
        "tz": TIME_SPEC[ds]["tz"], "time_column_means": TIME_SPEC[ds]["column_means"],
        "rhythm_trough_hour": st["argmin_hour"], "rhythm_day_night_ratio": st["day_night_ratio"],
        "num_pois_incl_pad": num_pois,
        "data_fingerprint": fingerprint(splits, num_pois),
    }
    print(f"[official] users={n_users:,} POIs={n_train_pois:,} check-ins={n_check:,} "
          f"trajectories={n_traj:,}")
    print(f"[official] local time OK (trough {st['argmin_hour']:02d}:00, "
          f"day/night {st['day_night_ratio']:.1f}) via {TIME_SPEC[ds]['tz']}, "
          f"column read as {TIME_SPEC[ds]['column_means']}")
    print(f"[official] fingerprint {stats['data_fingerprint']}")

    # Time origin. The model only ever uses DIFFERENCES of timestamps, so a
    # constant offset cancels exactly; measuring hours from the dataset's own
    # start instead of from 1970 keeps the magnitude near 1e4 rather than 1e9.
    # That is what lets the tensors be float32 without reintroducing the 128 s
    # quantization bug (float32 ulp at 1.3e9 s is 128 s; at 1.3e4 h it is 3.5 s).
    # It also makes the code run on MPS, which cannot hold float64 at all.
    t_ref = min(c["ts_utc"] for s_ in splits.values() for t in s_.values() for c in t)

    return {
        "t_ref": float(t_ref),
        "train_data": splits["train"], "val_data": splits["val"], "test_data": splits["test"],
        "poi_locations": poi_locations, "poi_categories": poi_categories,
        "poi_popularity": popularity, "num_pois": num_pois, "stats": stats,
        "default_lat": float(train_df["Latitude"].mean()),
        "default_lon": float(train_df["Longitude"].mean()),
    }


Writing castpoi/official.py


In [ ]:
%%writefile castpoi/data.py
"""Torch datasets and dataloaders.

No preprocessing happens here. Filtering and splitting come from the official
files loaded by official.py and from build_loo_split.py; see DATA.md. This module
only turns those trajectories into batched tensors.
"""
import math
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


class DataUnavailable(RuntimeError):
    """Kept for API compatibility. Real data errors now raise OfficialDataMissing."""


def _pack_repeat_history(history: List[Dict], max_len: int) -> List[int]:
    """POI ids over a long window, used only by the revisit features.

    Kept separate from the sequence window: attention is O(L) per candidate, so
    the reader truncates to max_history_len, but counting visits is cheap and a
    longer window classifies more targets correctly as revisits.
    """
    return [c["poi_idx"] for c in history[-max_len:]]


def _pack(history: List[Dict], max_len: int, default_lat: float, default_lon: float,
          t_ref: float = 0.0):
    """Truncate to the last max_len, then LEFT-pad to a fixed width.

    Timestamps come out as HOURS SINCE t_ref, not raw epoch seconds. See
    official.load_official for why: differences are all the model uses, and the
    smaller magnitude is what makes float32 safe (and MPS possible)."""
    h = history[-max_len:]
    n = len(h)
    pad = max_len - n
    poi = [0] * pad + [c["poi_idx"] for c in h]
    ts = [(h[0]["ts_utc"] - t_ref) / 3600.0] * pad + [(c["ts_utc"] - t_ref) / 3600.0 for c in h]
    hour = [0.0] * pad + [c["hour"] for c in h]
    dow = [0] * pad + [c["dow"] for c in h]
    loc = [[default_lat, default_lon]] * pad + [[c["latitude"], c["longitude"]] for c in h]
    return poi, ts, hour, dow, loc, n


def _tensors(poi, ts, hour, dow, loc, n, target, t_ref=0.0, rep_hist=None, rep_len=0):
    d = {
        "poi_ids": torch.tensor(poi, dtype=torch.long),
        "ts_hours": torch.tensor(ts, dtype=torch.float32),   # hours since t_ref
        "hour": torch.tensor(hour, dtype=torch.float32),
        "dow": torch.tensor(dow, dtype=torch.long),
        "locations": torch.tensor(loc, dtype=torch.float32),
        "seq_len": torch.tensor(n, dtype=torch.long),
        "target_poi": torch.tensor(target["poi_idx"], dtype=torch.long),
        "target_hour": torch.tensor(target["hour"], dtype=torch.float32),
        "target_dow": torch.tensor(target["dow"], dtype=torch.long),
        "target_location": torch.tensor([target["latitude"], target["longitude"]], dtype=torch.float32),
    }
    if rep_hist is not None:
        pad = rep_len - len(rep_hist)
        d["repeat_hist"] = torch.tensor([0] * pad + rep_hist, dtype=torch.long)
    return d


class POITrainDataset(Dataset):
    def __init__(self, train_data, num_pois, poi_popularity, config,
                 default_lat=0.0, default_lon=0.0, t_ref=0.0):
        self.num_pois = num_pois
        self.t_ref = t_ref
        # Under train_objective="full" nothing consumes neg_ids, and sampling
        # them anyway would cost ~200 ms per batch of 512 for a tensor the
        # training step throws away.
        self.sample_negatives = config.get("train_objective", "sampled") != "full"
        self.num_negatives = config["num_negatives"]
        self.max_history_len = config["max_history_len"]
        self.repeat_history_len = config.get("repeat_history_len", 512)
        self.default_lat, self.default_lon = default_lat, default_lon
        self.valid_idx = np.where(poi_popularity > 0)[0]
        p = poi_popularity[self.valid_idx]
        self.valid_probs = p / p.sum()
        # np.random.choice(p=...) rebuilds an O(|V|) cumulative sum on EVERY call.
        # At 499 negatives x ~82k samples per epoch that dominated the whole
        # training step: measured 38 s/epoch of pure data loading on a fast CPU,
        # which on a 2-vCPU Colab runtime left the GPU idle ~75% of the time.
        # Build the CDF once; sample with searchsorted, same distribution.
        self._cdf = np.cumsum(self.valid_probs)
        self._cdf[-1] = 1.0

        self.samples = []
        for uid, traj in train_data.items():
            seen = set()
            for i in range(1, len(traj)):
                seen.add(traj[i - 1]["poi_idx"])
                self.samples.append({"history": traj[:i], "target": traj[i],
                                     "is_explore": traj[i]["poi_idx"] not in seen})

    def __len__(self):
        return len(self.samples)

    def _sample_popularity(self, size: int) -> np.ndarray:
        """Popularity-weighted draw via the prebuilt CDF. Same distribution as
        np.random.choice(p=self.valid_probs), without its per-call O(|V|) setup."""
        return self.valid_idx[np.searchsorted(self._cdf, np.random.random(size))]

    def _sample_negatives(self, target: int) -> List[int]:
        """Half popularity-weighted, half uniform, de-duplicated, target excluded.

        Plain Python with a set: at these sizes (~500 draws over a 5k vocabulary)
        a vectorised numpy version measured slower. Costs roughly 200 ms per batch
        of 512, which num_workers > 0 hides.
        """
        n_pop = self.num_negatives // 2
        neg = set()
        for _ in range(8):
            if len(neg) >= n_pop:
                break
            neg.update(int(c) for c in self._sample_popularity(n_pop * 2) if c != target)
            if len(neg) > n_pop:
                neg = set(list(neg)[:n_pop])
        for _ in range(8):
            if len(neg) >= self.num_negatives:
                break
            for c in np.random.randint(1, self.num_pois, size=(self.num_negatives - len(neg)) * 2):
                if c != target:
                    neg.add(int(c))
                if len(neg) >= self.num_negatives:
                    break
        out = list(neg)[: self.num_negatives]
        while len(out) < self.num_negatives:
            r = int(np.random.randint(1, self.num_pois))
            if r != target:
                out.append(r)
        return out

    def __getitem__(self, idx):
        s = self.samples[idx]
        packed = _pack(s["history"], self.max_history_len, self.default_lat,
                       self.default_lon, self.t_ref)
        d = _tensors(*packed, s["target"], self.t_ref,
                     _pack_repeat_history(s["history"], self.repeat_history_len),
                     self.repeat_history_len)
        if self.sample_negatives:
            d["neg_ids"] = torch.tensor(self._sample_negatives(s["target"]["poi_idx"]), dtype=torch.long)
        d["is_explore"] = torch.tensor(float(s["is_explore"]), dtype=torch.float32)
        return d


class POIEvalDataset(Dataset):
    def __init__(self, history_base: Dict, eval_data: Dict, config,
                 default_lat=0.0, default_lon=0.0, t_ref=0.0):
        self.max_history_len = config["max_history_len"]
        self.repeat_history_len = config.get("repeat_history_len", 512)
        self.t_ref = t_ref
        self.default_lat, self.default_lon = default_lat, default_lon
        self.samples = []
        for uid, traj in eval_data.items():
            base = history_base.get(uid, [])
            for i in range(len(traj)):
                history = base + traj[:i]
                if history:
                    self.samples.append({"history": history, "target": traj[i]})

    def __len__(self):
        return len(self.samples)

    @property
    def check_ins_ids(self) -> np.ndarray:
        """Official check_ins_id of each sample's target, in sample order.

        This is what lets a rank vector from this repo be joined to one from a
        foreign implementation of the same task.

        Sample order is a permutation of test-CSV row order in general: the loop
        above walks users, and any user whose first eval check-in has no prior
        history contributes no sample at all. On the official NYC files it happens
        to come out as the identity -- 9,074 samples, 9,074 rows, same sequence --
        because those files arrive sorted by (UserId, ts_utc) and every test user
        already has training history. That is a property of the data, not of this
        code, so do not index into the CSV by position; join on the id.

        Derived from self.samples rather than re-walking eval_data so it stays a
        projection of the thing it labels: if the loop above changes, this follows
        instead of quietly disagreeing with it.
        """
        return np.array([s["target"]["check_ins_id"] for s in self.samples], dtype=np.int64)

    def __getitem__(self, idx):
        s = self.samples[idx]
        packed = _pack(s["history"], self.max_history_len, self.default_lat,
                       self.default_lon, self.t_ref)
        return _tensors(*packed, s["target"], self.t_ref,
                        _pack_repeat_history(s["history"], self.repeat_history_len),
                        self.repeat_history_len)


def collate_fn(batch):
    keys = ["poi_ids", "ts_hours", "hour", "dow", "locations", "target_poi",
            "target_hour", "target_dow", "target_location"]
    if "repeat_hist" in batch[0]:
        keys = keys + ["repeat_hist"]
    out = {k: torch.stack([b[k] for b in batch]) for k in keys}
    out["seq_lengths"] = torch.stack([b["seq_len"] for b in batch])
    for k in ("neg_ids", "is_explore"):
        if k in batch[0]:
            out[k] = torch.stack([b[k] for b in batch])
    return out


def _worker_init(worker_id: int) -> None:
    """Give each dataloader worker its own numpy stream.

    torch seeds each worker's `torch` RNG but NOT numpy's. Our negative sampler
    is pure numpy, so without this every worker would draw the SAME negatives --
    a silent correctness bug that only appears once num_workers > 0.
    """
    seed = (torch.initial_seed() + worker_id) % (2 ** 32)
    np.random.seed(seed)


def create_dataloaders(data: Dict, config: Dict, num_workers: int = 0):
    t_ref = data.get("t_ref", 0.0)
    train_ds = POITrainDataset(data["train_data"], data["num_pois"], data["poi_popularity"],
                               config, data["default_lat"], data["default_lon"], t_ref)
    val_ds = POIEvalDataset(data["train_data"], data["val_data"], config,
                            data["default_lat"], data["default_lon"], t_ref)
    train_plus_val = {uid: traj + data["val_data"].get(uid, [])
                      for uid, traj in data["train_data"].items()}
    test_ds = POIEvalDataset(train_plus_val, data["test_data"], config,
                             data["default_lat"], data["default_lon"], t_ref)

    ebs = config.get("eval_batch_size", config["batch_size"])
    mk = lambda ds, bs, sh: DataLoader(ds, batch_size=bs, shuffle=sh, collate_fn=collate_fn,
                                       num_workers=num_workers,
                                       worker_init_fn=_worker_init if num_workers else None,
                                       persistent_workers=num_workers > 0,
                                       pin_memory=torch.cuda.is_available())
    print(f"[data] samples: train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}")
    return (mk(train_ds, config["batch_size"], True), mk(val_ds, ebs, False), mk(test_ds, ebs, False))


Writing castpoi/data.py


In [ ]:
%%writefile castpoi/layers.py
"""Shared encoders and losses used by the CaST-POI ranker."""
import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_r, lon1_r = torch.deg2rad(lat1), torch.deg2rad(lon1)
    lat2_r, lon2_r = torch.deg2rad(lat2), torch.deg2rad(lon2)
    dlat, dlon = lat2_r - lat1_r, lon2_r - lon1_r
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1_r) * torch.cos(lat2_r) * torch.sin(dlon / 2) ** 2
    return R * 2 * torch.asin(torch.sqrt(torch.clamp(a, 0, 1)))


class TemporalEncoding(nn.Module):
    """Periodic time features from local hour-of-day and day-of-week plus a slot embedding."""

    def __init__(self, slot_embed_dim: int = 16, num_slots: int = 4, dropout: float = 0.1):
        super().__init__()
        self.omega_h = 2 * math.pi / 24.0
        self.omega_w = 2 * math.pi / 7.0
        self.slot_embedding = nn.Embedding(num_slots, slot_embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = 4 + slot_embed_dim
        nn.init.normal_(self.slot_embedding.weight, std=0.02)

    @staticmethod
    def time_slot(hour: torch.Tensor) -> torch.Tensor:
        slot = torch.full(hour.shape, 3, dtype=torch.long, device=hour.device)
        slot[(hour >= 6) & (hour < 12)] = 0
        slot[(hour >= 12) & (hour < 18)] = 1
        slot[hour >= 18] = 2
        return slot

    def forward(self, hour: torch.Tensor, dow: torch.Tensor) -> torch.Tensor:
        h, w = hour.float(), dow.float()
        feats = torch.stack([
            torch.sin(self.omega_h * h), torch.cos(self.omega_h * h),
            torch.sin(self.omega_w * w), torch.cos(self.omega_w * w),
        ], dim=-1)
        return self.dropout(torch.cat([feats, self.slot_embedding(self.time_slot(h))], dim=-1))


class SpatialEncoding(nn.Module):
    """Displacement features from the previous POI: log distance, a bucket embedding, and dlat/dlon."""

    DIST_BUCKETS = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, float("inf")]

    def __init__(self, output_dim: int = 32, num_dist_buckets: int = 8,
                 dist_embed_dim: int = 16, dropout: float = 0.1):
        super().__init__()
        self.num_dist_buckets = num_dist_buckets
        self.dist_embedding = nn.Embedding(num_dist_buckets, dist_embed_dim)
        self.register_buffer("_edges", torch.tensor(self.DIST_BUCKETS[1:-1]), persistent=False)
        self.projection = nn.Sequential(
            nn.Linear(1 + dist_embed_dim + 2, output_dim), nn.ReLU(),
            nn.Linear(output_dim, output_dim), nn.Dropout(dropout),
        )
        self.output_dim = output_dim
        nn.init.normal_(self.dist_embedding.weight, std=0.02)

    def _bucket(self, d: torch.Tensor) -> torch.Tensor:
        return torch.bucketize(d, self._edges).clamp_(max=self.num_dist_buckets - 1)

    def forward(self, locations: torch.Tensor, prev_locations: torch.Tensor) -> torch.Tensor:
        lat1, lon1 = prev_locations[..., 0], prev_locations[..., 1]
        lat2, lon2 = locations[..., 0], locations[..., 1]
        dist = haversine_distance(lat1, lon1, lat2, lon2)
        feats = torch.cat([
            torch.log1p(dist).unsqueeze(-1),
            self.dist_embedding(self._bucket(dist)),
            ((lat2 - lat1) / 0.1).unsqueeze(-1),
            ((lon2 - lon1) / 0.1).unsqueeze(-1),
        ], dim=-1)
        return self.projection(feats)


class InputEncoder(nn.Module):
    """POI embedding concatenated with temporal and spatial encodings."""

    def __init__(self, num_pois, poi_embed_dim, slot_embed_dim, spatial_dim,
                 num_dist_buckets, dist_embed_dim, dropout):
        super().__init__()
        self.poi_embedding = nn.Embedding(num_pois, poi_embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.poi_embedding.weight)
        with torch.no_grad():
            self.poi_embedding.weight[0].zero_()
        self.temporal_enc = TemporalEncoding(slot_embed_dim, dropout=dropout)
        self.spatial_enc = SpatialEncoding(spatial_dim, num_dist_buckets, dist_embed_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = poi_embed_dim + self.temporal_enc.output_dim + spatial_dim

    def forward(self, poi_ids, hour, dow, locations, prev_locations):
        return self.dropout(torch.cat([
            self.poi_embedding(poi_ids),
            self.temporal_enc(hour, dow),
            self.spatial_enc(locations, prev_locations),
        ], dim=-1))


def repeat_features(poi_ids: torch.Tensor, num_pois: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Per-candidate revisit signals from the visible history: log visit count, recency, visited flag.

    History is left-padded with 0; index 0 is the padding slot and is zeroed out.
    """
    B, L = poi_ids.shape
    valid = (poi_ids != 0).to(torch.float32)

    counts = torch.zeros(B, num_pois, device=poi_ids.device, dtype=torch.float32)
    counts.scatter_add_(1, poi_ids, valid)
    counts[:, 0] = 0.0

    pos = torch.arange(1, L + 1, device=poi_ids.device, dtype=torch.float32) / L
    pos = pos.unsqueeze(0).expand(B, L) * valid
    recency = torch.zeros(B, num_pois, device=poi_ids.device, dtype=torch.float32)
    recency.scatter_reduce_(1, poi_ids, pos, reduce="amax", include_self=True)
    recency[:, 0] = 0.0

    return torch.log1p(counts), recency, (counts > 0).to(torch.float32)


class RepeatGate(nn.Module):
    """Map the sequence state to three (unconstrained) weights over the revisit features."""

    def __init__(self, in_dim: int, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, 3))
        nn.init.zeros_(self.net[-1].bias)
        nn.init.normal_(self.net[-1].weight, std=0.01)

    def forward(self, x):
        return self.net(x)


class CELoss(nn.Module):
    """Cross-entropy with label smoothing and a hard-negative BPR margin.

    forward() scores a sampled candidate set; forward_full() scores the full vocabulary.
    """

    def __init__(self, label_smoothing=0.02, explore_weight=1.5, bpr_weight=0.5, margin=1.0):
        super().__init__()
        self.label_smoothing = label_smoothing
        self.explore_weight = explore_weight
        self.bpr_weight = bpr_weight
        self.margin = margin

    def forward(self, pos_scores, neg_scores, is_explore: Optional[torch.Tensor] = None):
        logits = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        B, C = logits.shape
        log_probs = F.log_softmax(logits, dim=1)
        if self.label_smoothing > 0:
            smooth = self.label_smoothing / C
            target = torch.full_like(log_probs, smooth)
            target[:, 0] = 1.0 - self.label_smoothing + smooth
        else:
            target = torch.zeros_like(log_probs)
            target[:, 0] = 1.0
        ce = -(target * log_probs).sum(1)

        k = min(10, neg_scores.size(1))
        hard_neg, _ = neg_scores.topk(k, dim=1)
        bpr = F.relu(self.margin - (pos_scores.unsqueeze(1) - hard_neg)).mean(1)

        loss = ce + self.bpr_weight * bpr
        if is_explore is not None and self.explore_weight > 1.0:
            loss = loss * (1.0 + (self.explore_weight - 1.0) * is_explore)
        return loss.mean()

    def forward_full(self, logits, target_ids, is_explore: Optional[torch.Tensor] = None):
        B, V = logits.shape
        log_probs = F.log_softmax(logits, dim=1)
        tgt_col = target_ids.unsqueeze(1)

        target = torch.zeros_like(log_probs)
        if self.label_smoothing > 0:
            smooth = self.label_smoothing / (V - 1)
            target[:, 1:] = smooth
            target.scatter_(1, tgt_col, 1.0 - self.label_smoothing + smooth)
        else:
            target.scatter_(1, tgt_col, 1.0)
        ce = -(target * log_probs).sum(1)

        masked = logits.scatter(1, tgt_col, float("-inf"))
        masked[:, 0] = float("-inf")
        k = min(10, V - 2)
        hard_neg, _ = masked.topk(k, dim=1)
        pos_scores = logits.gather(1, tgt_col).squeeze(1)
        bpr = F.relu(self.margin - (pos_scores.unsqueeze(1) - hard_neg)).mean(1)

        loss = ce + self.bpr_weight * bpr
        if is_explore is not None and self.explore_weight > 1.0:
            loss = loss * (1.0 + (self.explore_weight - 1.0) * is_explore)
        return loss.mean()




Writing castpoi/model.py


In [ ]:
%%writefile castpoi/engine.py
"""Training and evaluation loops.

Per-epoch history, per-sample test ranks and timing are written to disk so that
every reported number can be recomputed from a file rather than from memory.
"""
import math
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

from .metrics import metrics_from_ranks, per_sample_ranks, summarize, format_metrics


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs: int, total_epochs: int, eta_min: float = 1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.eta_min = eta_min
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]
        self.epoch = 0

    def step(self):
        self.epoch += 1
        if self.epoch <= self.warmup_epochs:
            factor = self.epoch / max(self.warmup_epochs, 1)
        else:
            progress = (self.epoch - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
            factor = 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
        for pg, base in zip(self.optimizer.param_groups, self.base_lrs):
            pg["lr"] = self.eta_min + (base - self.eta_min) * factor

    def get_lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]


def _query_spatial_context(batch: Dict[str, torch.Tensor]):
    """Current position and previous position, both taken from the history.

    An earlier version passed `target_location`, the coordinates of the POI being
    predicted. SpatialEncoding turns that into the displacement to the answer,
    which is future information; results produced before protocol_rev 2 carry
    that leak. Sequences are left-padded, so column -1 is the most recent real
    check-in and -2 the one before it.

    A user with a single check-in of history has no column -2, only padding filled
    with the dataset mean coordinate. Those users get prev := current, i.e. zero
    displacement, rather than a fictitious move from the centroid.
    """
    cur = batch["locations"][:, -1, :]
    prev = batch["locations"][:, -2, :]
    has_prev = (batch["seq_lengths"] >= 2).unsqueeze(-1).to(cur.dtype)
    return cur, prev * has_prev + cur * (1 - has_prev)


def _to_device(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def _forward(model, batch):
    return model(
        poi_ids=batch["poi_ids"],
        ts_hours=batch["ts_hours"],
        hour=batch["hour"],
        dow=batch["dow"],
        locations=batch["locations"],
        seq_lengths=batch["seq_lengths"],
        query_hour=batch["target_hour"],
        query_dow=batch["target_dow"],
        query_location=_query_spatial_context(batch)[0],
        prev_query_location=_query_spatial_context(batch)[1],
        repeat_hist=batch.get("repeat_hist"),
    )


def train_epoch(model, loader, optimizer, criterion, device, config) -> float:
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = _to_device(batch, device)
        h_proj, _, rep = _forward(model, batch)
        if config.get("train_objective", "sampled") == "full":
            # Same scoring path evaluation uses, so training and testing now pose
            # the model the identical |L|-way problem.
            logits = model.compute_all_scores(h_proj, rep)
            loss = criterion.forward_full(logits, batch["target_poi"], batch.get("is_explore"))
        else:
            pos, neg = model.compute_sampled_scores(h_proj, batch["target_poi"], batch["neg_ids"], rep)
            loss = criterion(pos, neg, batch.get("is_explore"))
        if not torch.isfinite(loss):
            raise RuntimeError(
                "training loss became NaN/Inf. Refusing to continue: a diverged model "
                "scores NaN, and NaN ranks as position 1 for every sample, which would "
                "be reported as a flawless 100% HR@k. Lower the learning rate, or check "
                "the device (Apple MPS has produced NaN here where CPU and CUDA do not).")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if config["gradient_clip"] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"])
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(n, 1)


TOPK_KEEP = 20


@torch.no_grad()
def evaluate(model, loader, device, ks=(5, 10, 20), collect_alpha: bool = False,
             collect_topk: bool = False):
    """Returns (summary, per_sample_dict, extras). per_sample arrays enable paired tests."""
    model.eval()
    ranks, alphas, targets, topk = [], [], [], []
    for batch in loader:
        batch = _to_device(batch, device)
        h_proj, alpha, rep = _forward(model, batch)
        scores = model.compute_all_scores(h_proj, rep)
        ranks.append(per_sample_ranks(scores, batch["target_poi"]).cpu().numpy())
        targets.append(batch["target_poi"].cpu().numpy())
        # Rank-based metrics are recoverable from `ranks` alone, but coverage,
        # novelty and popularity bias need the predicted ids. Off by default
        # because validation runs this every epoch and discards the result;
        # run.py enables it for the single test evaluation that is saved.
        if collect_topk:
            topk.append(scores.topk(min(TOPK_KEEP, scores.size(1)),
                                    dim=1).indices.cpu().numpy().astype(np.int32))
        if collect_alpha:
            alphas.append(alpha.cpu().numpy())

    ranks = np.concatenate(ranks).astype(np.float64)
    per_sample = metrics_from_ranks(ranks, ks)
    extras = {"ranks": ranks, "targets": np.concatenate(targets),
              "topk": np.concatenate(topk) if topk else None}

    # Official check_ins_id per sample, so these ranks can be joined to a foreign
    # implementation's. Position i of `ranks` means sample i of the dataset only
    # because the eval loaders are built with shuffle=False; under a shuffling
    # sampler the join would still run and would pair every sample with the wrong
    # id, so check rather than assume.
    ds = getattr(loader, "dataset", None)
    if hasattr(ds, "check_ins_ids"):
        if not isinstance(loader.sampler, torch.utils.data.SequentialSampler):
            raise RuntimeError(
                f"eval loader uses {type(loader.sampler).__name__}, not SequentialSampler. "
                f"Rank i would not correspond to sample i, so check_ins_id would mislabel "
                f"every row and any paired test built on it would be silently wrong.")
        cids = ds.check_ins_ids
        if len(cids) != len(ranks):
            raise RuntimeError(f"{len(cids)} check_ins_ids vs {len(ranks)} ranks.")
        extras["check_ins_id"] = cids
    if collect_alpha:
        extras["alpha"] = np.concatenate(alphas, axis=0)
    return summarize(per_sample), per_sample, extras


def train_model(model, train_loader, val_loader, config, device, logger=None) -> Tuple[nn.Module, Dict]:
    """Train with warmup+cosine LR and early stopping on validation HR@10."""
    from .layers import CELoss

    log = (logger.info if logger else print)
    model = model.to(device)
    criterion = CELoss(config["label_smoothing"], config["explore_weight"],
                           config.get("bpr_weight", 0.5), config.get("bpr_margin", 1.0))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"],
                                  weight_decay=config["weight_decay"])
    scheduler = WarmupCosineScheduler(optimizer, config["warmup_epochs"], config["num_epochs"])

    history = {"epochs": [], "best_epoch": None, "stopped_early": False}
    best_hr10, best_state, patience = -1.0, None, 0
    t_start = time.time()

    for epoch in range(1, config["num_epochs"] + 1):
        t0 = time.time()
        loss = train_epoch(model, train_loader, optimizer, criterion, device, config)
        train_s = time.time() - t0
        scheduler.step()

        t1 = time.time()
        val_metrics, _, _ = evaluate(model, val_loader, device, config["eval_ks"])
        eval_s = time.time() - t1

        history["epochs"].append({
            "epoch": epoch, "train_loss": loss, "lr": scheduler.get_lr(),
            "train_seconds": train_s, "eval_seconds": eval_s,
            "val": val_metrics,
        })
        log(f"epoch {epoch:3d} | loss {loss:.4f} | lr {scheduler.get_lr():.2e} | "
            f"{train_s:.1f}s+{eval_s:.1f}s | val {format_metrics(val_metrics)}")

        improved = val_metrics["HR@10"] > best_hr10
        if improved:
            best_hr10 = val_metrics["HR@10"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            patience = 0
        elif epoch > config["warmup_epochs"]:
            patience += 1
            if patience >= config["early_stopping_patience"]:
                log(f"early stop at epoch {epoch} (best val HR@10 {best_hr10 * 100:.2f} @ epoch {history['best_epoch']})")
                history["stopped_early"] = True
                break

    history["total_train_seconds"] = time.time() - t_start
    history["best_val_hr10"] = best_hr10
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def measure_inference(model, loader, device, batch_size: int, n_warmup: int = 3, n_iters: int = 20) -> Dict:
    """Honest latency/throughput: measured separately, not derived one from the other."""
    model.eval()
    batch = _to_device(next(iter(loader)), device)
    B = batch["poi_ids"].size(0)

    for _ in range(n_warmup):
        h, _, rep = _forward(model, batch)
        model.compute_all_scores(h, rep)
    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        h, _, rep = _forward(model, batch)
        model.compute_all_scores(h, rep)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    per_batch_ms = float(times.mean() * 1000)
    out = {
        "batch_size": B,
        "batch_latency_ms_mean": per_batch_ms,
        "batch_latency_ms_std": float(times.std(ddof=1) * 1000) if len(times) > 1 else 0.0,
        "per_query_latency_ms": per_batch_ms / B,
        "throughput_queries_per_s": B / (per_batch_ms / 1000),
        "n_iters": n_iters,
        "device": str(device),
    }
    if device.type == "cuda":
        out["peak_memory_gb"] = torch.cuda.max_memory_allocated() / 1024 ** 3
    return out




Writing castpoi/engine.py


In [ ]:
%%writefile run_recbole.py
#!/usr/bin/env python3
"""Run a RecBole sequential baseline on OUR data and write a run in OUR format.

This is the batch-A path for Table 1's sequential baselines (SASRec, BERT4Rec,
DuoRec, ...): use RecBole's faithful official implementations, but on the exact
test targets CaST-POI is evaluated on, scored full-vocabulary, and aligned
sample-for-sample by check_ins_id. The output is a run directory identical in
shape to run_experiment.py's, so the verdict table and the revisit/new-POI split
pick these models up with no extra code.

Verified end to end locally before this was written: RecBole benchmark mode loads
our pre-split, pre-augmented samples; item_num == |L|+1; the extracted ranks align
9,074/9,074 to our test set by check_ins_id.

Two things make it comparable rather than merely "RecBole ran":
  - ranks are reordered into OUR POIEvalDataset order and the tie convention is
    OUR metrics.per_sample_ranks (mid-rank), so ranks.npy is directly comparable
    to every other ranks.npy in runs/.
  - the input history is capped at max_history_len, the same budget our own runs
    used, so the transformers do not get a longer context than we gave ours.

    python run_recbole.py --dataset nyc --model SASRec --lr 1e-3 --tag tune
"""
import argparse
import hashlib
import json
import sys
import tempfile
import time
from pathlib import Path

import numpy as np

# RecBole 1.2.1 predates NumPy 2.0 and its own compatibility shim references
# aliases NumPy 2 removed. Restore them before RecBole imports, or Config() dies
# on `np.float = np.float_`. Harmless on NumPy 1.x (the attrs already exist).
import numpy as _np
for _old, _new in [("float_", "float64"), ("complex_", "complex128"),
                   ("unicode_", "str_"), ("bool8", "bool_")]:
    if not hasattr(_np, _old):
        setattr(_np, _old, getattr(_np, _new))

from castpoi.official import load_official
from castpoi.config import resolve_config, BASE_CONFIG
from castpoi.metrics import REPORT_KS, per_sample_ranks, metrics_from_ranks, summarize

MAX_LEN = BASE_CONFIG["max_history_len"]
TOPK_KEEP = 20        # == castpoi.engine.TOPK_KEEP; keeps topk.npy comparable


def build_samples(data):
    """Our POITrain/POIEvalDataset targets, keeping user_id and check_ins_id.
    Returns dict split -> list of (uid, [prefix poi_idx], target_poi_idx, cid)."""
    train, val, test = data["train_data"], data["val_data"], data["test_data"]
    tpv = {u: t + val.get(u, []) for u, t in train.items()}
    out = {"train": [], "valid": [], "test": []}

    def emit(split, base_map, traj_map):
        for uid, traj in traj_map.items():
            base = base_map.get(uid, []) if base_map is not None else []
            for i in range(len(traj)):
                hist = base + traj[:i]
                if not hist:
                    continue
                out[split].append((uid, [c["poi_idx"] for c in hist[-MAX_LEN:]],
                                   traj[i]["poi_idx"], traj[i]["check_ins_id"]))

    emit("train", None, train)
    emit("valid", train, val)
    emit("test", tpv, test)
    return out


def write_inter(samp, path):
    with open(path, "w") as f:
        f.write("user_id:token\titem_id_list:token_seq\titem_id:token\tcheck_ins_id:token\n")
        for uid, seq, tgt, cid in samp:
            f.write(f"{uid}\t{' '.join(map(str, seq))}\t{tgt}\t{cid}\n")


def recbole_ranks(dataset, model_name, workdir, lr, epochs, device, seed):
    """Train `model_name` on the exported data; return {check_ins_id: (rank, top20)}."""
    import torch
    from recbole.config import Config
    from recbole.data import create_dataset, data_preparation
    from recbole.utils import get_model, get_trainer, init_seed

    cfg = {
        "data_path": str(workdir),
        # RecBole only calls init_seed() inside quick_start, which we bypass to
        # avoid its ray import. Without this the seed is never set at all: every
        # run drew fresh OS entropy, `--seed` changed nothing but the output
        # directory, and the "seed" recorded in metrics.json was fiction. That is
        # survivable for one-off runs and fatal for a multi-seed campaign, where
        # the whole point is that the seed is the only thing that differs.
        "seed": seed,
        "reproducibility": True,
        "benchmark_filename": ["train", "valid", "test"],
        "load_col": {"inter": ["user_id", "item_id_list", "item_id", "check_ins_id"]},
        "alias_of_item_id": ["item_id_list"],   # history shares the item vocab
        "USER_ID_FIELD": "user_id", "ITEM_ID_FIELD": "item_id",
        "eval_args": {"mode": "full", "order": "TO"},
        "metrics": ["Recall", "MRR", "NDCG"], "topk": [5, 10],
        "valid_metric": "Recall@10",           # so LR is selected on val, like ours
        "MAX_ITEM_LIST_LENGTH": MAX_LEN,
        "train_neg_sample_args": None,          # sequential models use CE
        "epochs": epochs,
        "train_batch_size": 2048, "eval_batch_size": 4096,
        "device": device, "use_gpu": device != "cpu",
        "stopping_step": 10,
    }
    # lr=None means "use whatever this model's own RecBole config specifies",
    # which is the value RecBole curates from the original paper. That is the
    # faithful setting for a baseline: sweeping OUR grid over someone else's model
    # is neither their protocol nor a fair one.
    if lr is not None:
        cfg["learning_rate"] = lr

    config = Config(model=model_name, dataset=dataset, config_dict=cfg)
    init_seed(config["seed"], config["reproducibility"])
    ds = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, ds)
    # Seed again before constructing the model, exactly as quick_start does: the
    # first call covers dataset construction, this one covers weight init, and
    # skipping it would leave initialisation dependent on how much RNG the data
    # pipeline happened to consume.
    init_seed(config["seed"], config["reproducibility"])
    model = get_model(config["model"])(config, train_data._dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
    best_val, _ = trainer.fit(train_data, valid_data, verbose=False, show_progress=False)

    model.eval()
    ITEM = config["ITEM_ID_FIELD"]
    cid_tokens = ds.field2id_token["check_ins_id"]   # RecBole idx -> original str
    # RecBole remaps item ids into its own contiguous space, so raw topk indices
    # are meaningless outside this process. Translate them back to OUR poi_idx or
    # topk.npy cannot be compared against any other run's. Index 0 is RecBole's
    # [PAD], which corresponds to our padding poi_idx 0.
    item_tokens = ds.field2id_token[ITEM]
    to_poi = np.array([0 if not str(t).lstrip("-").isdigit() else int(t) for t in item_tokens],
                      dtype=np.int32)
    keep = min(TOPK_KEEP, int(ds.item_num))
    out = {}
    with torch.no_grad():
        for batch in test_data:
            inter = batch[0].to(config["device"])
            scores = model.full_sort_predict(inter).view(-1, ds.item_num)
            # Column 0 is RecBole's [PAD], which is not a POI and must not compete.
            # RecBole does not mask it and our models do (_pad_mask = -1e9), so
            # leaving it in ranks RecBole against |L|+1 candidates while ranking
            # CaST-POI against |L| -- every sample where [PAD] outscores the target
            # costs RecBole one rank. Measured on NYC SASRec: [PAD] landed in the
            # top-10 for 287 of 9,074 samples. That asymmetry runs in OUR favour,
            # which is the direction that must never be left uncorrected.
            scores[:, 0] = -1e9
            # OUR tie convention, byte-for-byte the same call our models' ranks use
            r = per_sample_ranks(scores.cpu(), inter[ITEM].cpu()).numpy()
            tk = to_poi[scores.topk(keep, dim=1).indices.cpu().numpy()]
            cids = inter["check_ins_id"].cpu().numpy()
            for rank, c, row in zip(r, cids, tk):
                out[int(cid_tokens[c])] = (float(rank), row)
    return out, int(ds.item_num - 1), float(best_val), float(config["learning_rate"])


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--dataset", required=True, choices=["nyc", "tky", "ca"])
    p.add_argument("--model", required=True, help="a RecBole sequential model, e.g. SASRec / BERT4Rec / DuoRec")
    p.add_argument("--lr", type=float, default=None,
                   help="omit to use the model's own RecBole default, which is the "
                        "value curated from its original paper. Pass one only to "
                        "deliberately override a baseline's published setting.")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--tag", default="tune")
    p.add_argument("--out", default="runs", type=Path)
    p.add_argument("--official-dir", default=None)
    p.add_argument("--device", default="cuda")
    p.add_argument("--force", action="store_true")
    args = p.parse_args(argv)

    variant = f"{args.model.lower()}_recbole"
    lr_key = "default" if args.lr is None else f"{args.lr}"
    cfg_hash = hashlib.sha1(f"{args.model}|lr{lr_key}|ep{args.epochs}".encode()).hexdigest()[:10]
    run_dir = args.out / args.tag / args.dataset / variant / cfg_hash / f"seed{args.seed}"
    if (run_dir / "metrics.json").exists() and not args.force:
        print(f"[skip] already done: {run_dir}")
        return 0

    data = load_official(args.dataset, args.official_dir)
    samp = build_samples(data)

    t0 = time.time()
    with tempfile.TemporaryDirectory() as tmp:
        wd = Path(tmp) / args.dataset
        wd.mkdir(parents=True)
        for split, rows in samp.items():
            write_inter(rows, wd / f"{args.dataset}.{split}.inter")
        cid2rank, n_items, best_val, eff_lr = recbole_ranks(
            args.dataset, args.model, Path(tmp), args.lr, args.epochs, args.device, args.seed)
    train_seconds = time.time() - t0

    # Reorder into OUR POIEvalDataset test order so ranks.npy lines up with every
    # other run's ranks.npy (and with the revisit mask, which is positional).
    test_samples = samp["test"]
    missing = [cid for _, _, _, cid in test_samples if cid not in cid2rank]
    if missing:
        raise SystemExit(f"{len(missing)} of our test targets got no RecBole rank "
                         f"(first: {missing[:3]}). Alignment is broken; refusing to write.")
    ranks = np.array([cid2rank[cid][0] for _, _, _, cid in test_samples], dtype=np.float64)
    topk = np.stack([cid2rank[cid][1] for _, _, _, cid in test_samples]).astype(np.int32)
    targets = np.array([tgt for _, _, tgt, _ in test_samples], dtype=np.int64)

    # Compute the FULL metric set, not BASE_CONFIG["eval_ks"] (which is [5,10]).
    # ranks.npy makes every rank-based metric recomputable later, but putting them
    # in metrics.json now means no backfill pass is needed to build any table.
    test_metrics = summarize(metrics_from_ranks(ranks, REPORT_KS))

    run_dir.mkdir(parents=True, exist_ok=True)
    np.save(run_dir / "ranks.npy", ranks)
    np.save(run_dir / "targets.npy", targets)
    np.save(run_dir / "topk.npy", topk)   # in OUR poi_idx space, see recbole_ranks
    np.save(run_dir / "check_ins_id.npy",
            np.array([cid for _, _, _, cid in test_samples], dtype=np.int64))
    json.dump({
        "dataset": args.dataset, "variant": variant, "model": f"recbole:{args.model}",
        "seed": args.seed, "config_hash": cfg_hash,
        "config": {"learning_rate": eff_lr, "lr_source": "recbole-default" if args.lr is None else "override",
                   "epochs": args.epochs, "source": "RecBole", "recbole_seed": args.seed},
        "data_fingerprint": data["stats"]["data_fingerprint"],
        "test": test_metrics, "n_test_samples": int(len(ranks)),
        "num_candidate_items": n_items, "train_seconds": train_seconds,
        "best_val_hr10": best_val, "best_epoch": None, "epochs_run": args.epochs,
    }, open(run_dir / "metrics.json", "w"), indent=2)
    print(f"[{args.dataset}/{args.model}] HR@10={test_metrics['HR@10']*100:.2f} "
          f"MRR={test_metrics['MRR']*100:.2f} n={len(ranks)} -> {run_dir}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


Writing run_recbole.py


In [ ]:
%%writefile build_loo_split.py
#!/usr/bin/env python3
"""Build a per-user leave-one-out (LOO) split from the chronological data_official,
in the EXACT format castpoi/official.py reads, so CaST-POI and (via export) RecBole
run on one bit-identical LOO split.

Per user, chronological by UTCTimeOffset:
    last check-in     -> test
    second-to-last    -> validation
    all earlier       -> train
Users with < 3 check-ins are dropped (LOO needs >= 1 train + 1 val + 1 test).
val/test rows whose PoiId is not in the NEW train are removed (unseen-POI removal,
matching the upstream pipeline and the range assertion in official.py). Every
original column is preserved, so official.py loads it unchanged.

    python build_loo_split.py --src data_official --out data_loo --datasets nyc tky ca
"""
import argparse
import os
import pandas as pd

FILES = {"train": "train_sample.csv",
         "val":   "validate_sample_with_traj.csv",
         "test":  "test_sample_with_traj.csv"}


def build(ds, src, out):
    # 1. read + concat the three chronological splits (identical columns, verified)
    parts = [pd.read_csv(os.path.join(src, ds, fn), low_memory=False)
             for fn in FILES.values()]
    df = pd.concat(parts, ignore_index=True)
    n0 = len(df)

    # 2. chronological order per user. UTCTimeOffset is the wall-clock string; parse
    #    its first 19 chars. VERIFIED (2026-07-24, all 3 datasets): this yields the
    #    identical per-user ordering as the harness's ts_utc key (official.py) --
    #    the test target is the user's ts_utc-max check-in for 100% of users and
    #    there is ZERO future leakage (no history check-in has ts_utc > target). So
    #    a string sort is safe here (NYC DST introduces no mis-ordering) and no
    #    dependence on parse_times is needed. Stable mergesort keeps concat order
    #    (train,val,test) on exact-timestamp ties, matching official.py's own tie
    #    handling (its documented, negligible dup-timestamp edge case).
    df["_t"] = pd.to_datetime(df["UTCTimeOffset"].astype(str).str.slice(0, 19),
                              format="%Y-%m-%d %H:%M:%S", errors="coerce")
    assert df["_t"].notna().all(), f"{ds}: some UTCTimeOffset failed to parse"
    df = df.sort_values(["UserId", "_t"], kind="mergesort").reset_index(drop=True)

    # 3. LOO tag: rank from the end within each user (0 = last check-in)
    df["_rk"] = df.groupby("UserId").cumcount(ascending=False)
    df["_n"] = df.groupby("UserId")["UserId"].transform("size")
    df = df[df["_n"] >= 3].copy()
    df["_split"] = "train"
    df.loc[df["_rk"] == 0, "_split"] = "test"
    df.loc[df["_rk"] == 1, "_split"] = "val"

    # 4. unseen removal: drop val/test rows whose PoiId or UserId is absent from
    #    the new train (keeps official.py's range assertion satisfied).
    train_pois = set(df.loc[df["_split"] == "train", "PoiId"].unique())
    train_users = set(df.loc[df["_split"] == "train", "UserId"].unique())
    is_train = df["_split"] == "train"
    seen = df["PoiId"].isin(train_pois) & df["UserId"].isin(train_users)
    df = df[is_train | seen].copy()

    # 5. write ONLY the original CSV columns (drop the _t/_rk/_n/_split helpers) so
    #    official.py re-derives time from UTCTimeOffset itself, as for chronological.
    orig_cols = [c for c in df.columns if not c.startswith("_")]
    os.makedirs(os.path.join(out, ds), exist_ok=True)
    counts = {}
    for split, fn in FILES.items():
        sub = df.loc[df["_split"] == split, orig_cols]
        sub.to_csv(os.path.join(out, ds, fn), index=False)
        counts[split] = len(sub)

    n_users = df["UserId"].nunique()
    print(f"[{ds}] from {n0:,} check-ins -> train {counts['train']:,} / "
          f"val {counts['val']:,} / test {counts['test']:,}  "
          f"| users(with train) {n_users:,} | |L|~{len(train_pois):,}")
    print(f"[{ds}] LOO sanity: test ({counts['test']:,}) should be <= #users "
          f"({n_users:,}); one test target per user minus unseen-POI drops.")
    return counts, n_users


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="data_official")
    ap.add_argument("--out", default="data_loo")
    ap.add_argument("--datasets", nargs="+", default=["nyc", "tky", "ca"])
    a = ap.parse_args()
    for ds in a.datasets:
        build(ds, a.src, a.out)
    print("\nDone. Point the harness at it:  --official-dir", a.out)


if __name__ == "__main__":
    main()


Writing build_loo_split.py


In [ ]:
# === Build the per-user LOO split from data_official (verified locally: test == 1/user) ===
import subprocess
subprocess.run(['python', 'build_loo_split.py', '--src', DATA, '--out', 'data_loo',
                '--datasets', 'nyc', 'tky', 'ca'])


CompletedProcess(args=['python', 'build_loo_split.py', '--src', 'data_official', '--out', 'data_loo', '--datasets', 'nyc', 'tky', 'ca'], returncode=0)

In [ ]:
# === Run the RecBole sequential baselines under LOO (same split as CaST-POI) ===
# --lr omitted -> each model uses its own RecBole default (its published setting).
# run_recbole.py loads data_loo via load_official, builds the SAME test samples, and
# reorders ranks into OUR test order (aligned by check_ins_id) -> directly comparable.
import subprocess
# DATASETS = ['nyc', 'tky', 'ca']; SEEDS = [42, 43, 44]
DATASETS = ['nyc', 'tky', 'ca']; SEEDS = [43]
# MODELS = ['SASRec', 'BERT4Rec', 'CORE', 'NARM', 'Caser', 'SRGNN', 'STAMP']
MODELS = ['Caser']
for model in MODELS:
    for ds in DATASETS:
        for seed in SEEDS:
            cmd = ['python', 'run_recbole.py', '--dataset', ds, '--model', model,
                   '--seed', str(seed), '--tag', 'loo', '--official-dir', 'data_loo',
                   '--out', RUNS]
            print('>>>', ' '.join(cmd)); subprocess.run(cmd)


>>> python run_recbole.py --dataset nyc --model Caser --seed 43 --tag loo --official-dir data_loo --out /content/drive/MyDrive/castpoi/runs
>>> python run_recbole.py --dataset tky --model Caser --seed 43 --tag loo --official-dir data_loo --out /content/drive/MyDrive/castpoi/runs
>>> python run_recbole.py --dataset ca --model Caser --seed 43 --tag loo --official-dir data_loo --out /content/drive/MyDrive/castpoi/runs


In [ ]:
# === LOO leaderboard: RecBole baselines + CaST-POI (seed-averaged HR@10) ===
import glob, json, numpy as np
DATASETS = ['nyc', 'tky', 'ca']
def hr10(ds, variant):
    vals = []
    for m in glob.glob(f'{RUNS}/loo/{ds}/{variant}/*/seed*/metrics.json'):
        v = json.load(open(m))['test'].get('HR@10')
        if v is not None: vals.append(v * 100 if v <= 1.5 else v)
    return (np.mean(vals) if vals else float('nan')), len(vals)
ROWS = ['sasrec_recbole', 'bert4rec_recbole', 'core_recbole', 'narm_recbole',
        'caser_recbole', 'srgnn_recbole', 'stamp_recbole']
print(f"{'model':20}" + "".join(f"{d.upper():>9}" for d in DATASETS) + "   seeds")
for mv in ROWS:
    cells = [hr10(d, mv) for d in DATASETS]
    print(f"{mv:20}" + "".join(f"{c[0]:9.2f}" for c in cells)
          + "   " + "/".join(str(c[1]) for c in cells))
print("\\nHR@10 %, LOO. run_recbole aligns to CaST-POI's test order by check_ins_id, so these")
print("are directly comparable. All baselines run by us under one pipeline -- nothing cited.")


model                     NYC      TKY       CA   seeds
sasrec_recbole          56.99    56.02    44.25   3/3/3
bert4rec_recbole        54.69    53.67    36.80   3/3/3
core_recbole            57.76    53.48    39.92   3/3/3
narm_recbole            55.87    53.55    40.09   3/3/3
caser_recbole           54.15    51.81    34.48   3/3/1
srgnn_recbole           52.81    53.49    38.29   3/3/3
stamp_recbole           46.33    49.82    34.50   3/3/3
\nHR@10 %, LOO. run_recbole aligns to CaST-POI's test order by check_ins_id, so these
are directly comparable. All baselines run by us under one pipeline -- nothing cited.
